In [ ]:
from pathlib import Path
import sys

# Make this package's scripts/ importable (exp_info, regions, data_utils, ...).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "exp_info.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

In [ ]:
# Parameters: directories and settings for this run. Edit them for another
# machine or experiment, or override with papermill (-p NAME VALUE).
DATA_DIR = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/data"  # processed-data root
DIAG_DIR = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/diag_data"  # output diagnostic data

In [13]:
import os
import re
import numpy as np
import pandas as pd
import xarray as xr
from collections import OrderedDict
from typing import Optional, Tuple, List, Dict, Any, Set, Sequence

from matplotlib.colors import TwoSlopeNorm
import matplotlib as mpl
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
from matplotlib import path as mpath
import matplotlib.colors as mcolors
from matplotlib import cm

import cartopy.crs as ccrs
from matplotlib.ticker import FixedLocator
from matplotlib.colorbar import ColorbarBase
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Imports
from pathlib import Path


In [14]:
class ArcticAmplificationDiagnostics:
    """
    Streamed AA diagnostics for *monthly* inputs like:
      v3.LR.historical.en00.monthly_base.185001-205012.nc

    Pipeline per member:
      monthly -> seasonal/annual aggregation -> fixed-baseline anomalies ->
      β_global (over region_global), β_local (masked to region_local), trend (K/decade),
      AA map (β_local / β_global), and optional AA index.

    Highlights
    - target_period: analyze a subwindow (e.g., 1979–2018) without changing file discovery PERIOD.
    - region_global/region_local: explicit denominator/numerator regions.
    - Dask-friendly: opens with chunks="auto", guarantees time is single chunk for gufuncs.
    - Multi-season mode: open each member ONCE and compute multiple seasons to cut I/O.

    Note: This class is season-agnostic. Provide `season` explicitly to methods that need it.
    """

    def __init__(
        self,
        *,
        data_dir: str,
        group: str,
        nens: int,
        period: Tuple[int, int],                         # file discovery range (e.g., 1850–2050)
        target_period: Optional[Tuple[int, int]] = None, # analysis window (e.g., 1979–2018)
        baseline: Tuple[int, int] = (1981, 2010),
        var_name: str = "TS",
        lat_name: str = "lat",
        lon_name: str = "lon",
        time_coord: Optional[str] = None,
        aa_eps: float = 1e-5,
        # Regions (denominator/numerator):
        region_global: Optional[Dict[str, Tuple[float, float]]] = None,
        region_local:  Optional[Dict[str, Tuple[float, float]]] = None,
        region_subdir: str = "global",
        normalize_lon_to_180: bool = True,
        normalize_time_policy: str = "left",
        # I/O & performance:
        file_template: Optional[str] = None,  # e.g. "{group}.%(ENS).monthly_base.{y0m}-{y1m}.nc"
        verbose: bool = True,
        use_dask: bool = False,
        chunks: Optional[Dict[str, Any]] = None,  # kept for compatibility; not used at open
        # Output options:
        emit_aa_index: bool = True,  # area-weighted β_local(region_local)/β_global scalar
    ):
        self.data_dir = data_dir
        self.group = group
        self.nens = int(nens)
        self.period = (int(period[0]), int(period[1]))  # file search range
        self.analysis_period = tuple(target_period) if target_period else self.period
        self.baseline = (int(baseline[0]), int(baseline[1]))
        self.var_name = var_name
        self.lat_name = lat_name
        self.lon_name = lon_name
        self.time_coord = time_coord
        self.aa_eps = float(aa_eps)

        # Regions (defaults: global for denominator, Arctic for numerator)
        self.region_global = region_global or {"lat_range": (-90.0, 90.0), "lon_range": (-180.0, 180.0)}
        self.region_local  = region_local  or {"lat_range": ( 66.5, 90.0), "lon_range": (-180.0, 180.0)}
        self.region_subdir = region_subdir
        
        self.normalize_lon_to_180 = normalize_lon_to_180
        self.normalize_time_policy = str(normalize_time_policy).lower()

        self.verbose = bool(verbose)
        self.use_dask = bool(use_dask)
        self.chunks = chunks  # retained for API compatibility
        self.emit_aa_index = bool(emit_aa_index)

        if file_template is None:
            file_template = "{group}.%(ENS).monthly_base.{y0m}-{y1m}.nc"
        self.file_template = file_template

    def _log(self, msg: str):
        if self.verbose:
            print(msg)

    @staticmethod
    def _is_datetime64(da: xr.DataArray) -> bool:
        return np.issubdtype(da.dtype, np.datetime64)

    @staticmethod
    def _find_time_bnds(ds: xr.Dataset) -> Optional[xr.DataArray]:
        for name in ("time_bnds", "time_bounds", "bounds_time"):
            if name in ds.variables:
                return ds[name]
        return None

    @staticmethod
    def _ym_fix_for_first_of_next(y: int, m: int, d: int) -> Tuple[int, int]:
        if d != 1:
            return y, m
        m2 = m - 1
        return (y - 1, 12) if m2 == 0 else (y, m2)

    @staticmethod
    def _month_center_date(y: int, m: int, *, use_cftime=False, calendar=None):
        if use_cftime and _HAVE_CFTIME:
            cal = (calendar or "standard")
            ctor = getattr(
                cftime,
                "DatetimeNoLeap" if cal in ("noleap", "365_day") else
                "Datetime360Day" if cal == "360_day" else
                "DatetimeGregorian"
            )
            return ctor(int(y), int(m), 15)
        return np.datetime64(f"{int(y):04d}-{int(m):02d}-15", "ns")

    @staticmethod
    def _time_from_bounds(tb: xr.DataArray, policy: str = "center") -> xr.DataArray:
        if "nbnd" in tb.dims:
            left, right = tb.isel(nbnd=0), tb.isel(nbnd=1)
        elif "bnds" in tb.dims:
            left, right = tb.isel(bnds=0), tb.isel(bnds=1)
        else:
            left, right = tb[..., 0], tb[..., 1]
        if policy == "left":
            return left
        if policy == "right":
            return right
        return left + (right - left) / 2
            
    def _is_lon_0360(self, lon: xr.DataArray) -> bool:
        try:
            return float(lon.max()) > 180.0 and float(lon.min()) >= 0.0
        except Exception:
            return False
        
    def _clip_time_to_period(self, ds: xr.Dataset, tname: str) -> xr.Dataset:
        y0, y1 = self.period
        t = ds[tname].values
        if len(t) == 0:
            return ds

        # cftime?
        if hasattr(t[0], "year"):
            # assume noleap calendar (safe for your data)
            def month_len(y, m):
                if m in (1,3,5,7,8,10,12): return 31
                if m in (4,6,9,11):        return 30
                return 28
            ctor = type(t[0])
            start = ctor(y0, 1, 1)
            end   = ctor(y1, 12, month_len(y1, 12))
            mask = np.array([(v >= start) and (v <= end) for v in t], dtype=bool)
            return ds.isel({tname: mask})

        # datetime64 fallback
        start = np.datetime64(f"{y0:04d}-01-01")
        end   = np.datetime64(f"{y1:04d}-12-31")
        mask = (t >= start) & (t <= end)
        return ds.isel({tname: mask})

    def _normalize_time_monthly(self, ds: xr.Dataset, tname: str, policy: str = "center") -> xr.Dataset:
        t  = ds[tname]
        tb = self._find_time_bnds(ds)

        # ---- Prefer time_bnds: choose represented month, then place per policy ----
        if tb is not None:
            # left/right bounds
            if "nbnd" in tb.dims:
                left, right = tb.isel(nbnd=0), tb.isel(nbnd=1)
            elif "bnds" in tb.dims:
                left, right = tb.isel(bnds=0), tb.isel(bnds=1)
            else:
                left, right = tb[..., 0], tb[..., 1]

            L = left.values
            R = right.values

            # cftime-aware?
            cftime_mode = (len(L) > 0 and hasattr(L[0], "year"))
            if cftime_mode:
                # Represented month is the month of the LEFT bound (e.g., [Dec 1, Jan 1) => December)
                def month_len(y, m):
                    # noleap month lengths
                    if m in (1,3,5,7,8,10,12): return 31
                    if m in (4,6,9,11):        return 30
                    return 28  # February in noleap
                out = []
                for a, _b in zip(L, R):
                    ay, am = a.year, a.month
                    if policy == "left":
                        out.append(type(a)(ay, am, 1))
                    elif policy == "right":
                        out.append(type(a)(ay, am, month_len(ay, am)))
                    else:  # center
                        d15 = min(15, month_len(ay, am))
                        out.append(type(a)(ay, am, d15))
                return ds.assign_coords({tname: np.array(out, dtype=object)})
            else:
                # datetime64: use midpoint to pick month, then map per policy
                Lp = pd.to_datetime(L); Rp = pd.to_datetime(R)
                mid = Lp + (Rp - Lp) / 2
                if policy == "left":
                    new = mid.to_period("M").to_timestamp()                        # YYYY-MM-01
                elif policy == "right":
                    new = mid.to_period("M").to_timestamp() + pd.offsets.MonthEnd(0)
                else:
                    new = mid.to_period("M").to_timestamp() + pd.Timedelta(days=14)
                return ds.assign_coords({tname: new.to_numpy("datetime64[ns]")})

        # ---- No bounds: heuristic normalization (keep SAME month; no backshift) ----
        vals = t.values

        # cftime heuristic (noleap-safe)
        if len(vals) > 0 and hasattr(vals[0], "year"):
            def month_len(y, m):
                if m in (1,3,5,7,8,10,12): return 31
                if m in (4,6,9,11):        return 30
                return 28
            out = []
            for v in vals:
                y, m = v.year, v.month  # keep the timestamp's own month
                if policy == "left":
                    out.append(type(v)(y, m, 1))
                elif policy == "right":
                    out.append(type(v)(y, m, month_len(y, m)))
                else:  # center
                    out.append(type(v)(y, m, min(15, month_len(y, m))))
            return ds.assign_coords({tname: np.array(out, dtype=object)})

        # datetime64 heuristic
        if np.issubdtype(vals.dtype, "datetime64"):
            tt = pd.to_datetime(vals)
            if policy == "left":
                new = tt.to_period("M").to_timestamp()  # YYYY-MM-01
            elif policy == "right":
                new = tt.to_period("M").to_timestamp() + pd.offsets.MonthEnd(0)
            else:  # center
                new = tt.to_period("M").to_timestamp() + pd.Timedelta(days=14)
            return ds.assign_coords({tname: new.to_numpy("datetime64[ns]")})

        return ds


    def _coerce_lon_to_m180_p180(self, ds: xr.Dataset) -> xr.Dataset:
        """
        If lon is 0..360, convert to -180..180. For 1D lon, also sort along lon.
        Works for both 1D and 2D lon.
        """
        if self.lon_name not in ds:
            return ds

        lon = ds[self.lon_name]
        if not self._is_lon_0360(lon):
            return ds  # already -180..180 or mixed but <=180 max

        # Map to [-180,180)
        lon_new = ((lon + 180.0) % 360.0) - 180.0

        if lon.ndim == 1:
            # Assign new coord, then sort along lon so it’s monotonic
            ds2 = ds.assign_coords({self.lon_name: lon_new})
            ds2 = ds2.sortby(self.lon_name)
            return ds2
        else:
            # 2D lon grid: just replace values; no sort axis
            ds2 = ds.copy()
            ds2[self.lon_name] = lon_new
            return ds2

    # ---------- discovery ----------
    def discover_paths(self) -> List[str]:
        y0, y1 = self.period
        y0m, y1m = f"{y0}01", f"{y1}12"
        fmt = {"group": self.group, "y0": y0, "y1": y1, "y0m": y0m, "y1m": y1m}

        paths: List[str] = []
        for i in range(self.nens):
            ens = f"en{str(i).zfill(2)}"
            pat = self.file_template.format(**{k: fmt.get(k, "") for k in fmt}).replace("%(ENS)", ens)
            fullpat = os.path.join(self.data_dir, self.region_subdir, pat)

            if any(ch in fullpat for ch in "*?[]"):
                matches = sorted(glob.glob(fullpat))
                if matches:
                    paths.append(matches[0])
                else:
                    self._log(f"[WARN] missing (glob 0 hits): {fullpat}")
            else:
                if os.path.isfile(fullpat):
                    paths.append(fullpat)
                else:
                    self._log(f"[WARN] missing: {fullpat}")

        if not paths:
            raise FileNotFoundError("No ensemble files found.")
        self._log(f"[INFO] Found {len(paths)} ensemble members.")
        return paths

    # ---------- time helpers ----------
    def _detect_time_coord(self, ds: xr.Dataset) -> str:
        if self.time_coord:
            if self.time_coord in ds.coords or self.time_coord in ds.variables:
                return self.time_coord
            raise ValueError(f"time_coord='{self.time_coord}' not found in dataset.")
        for k in ("time", "year", "Time", "time_counter"):
            if k in ds.coords or k in ds.variables:
                self.time_coord = k
                return k
        for k in list(ds.coords) + list(ds.data_vars):
            if "time" in k.lower():
                self.time_coord = k
                return k
        raise ValueError("No time coordinate found (tried 'time', 'year', 'Time', 'time_counter', or fuzzy match).")

    @staticmethod
    def _year_array(tcoord: xr.DataArray) -> np.ndarray:
        vals = np.asarray(tcoord.values)
        if np.issubdtype(vals.dtype, np.number):
            return vals.astype(int)
        # datetime64 path
        if np.issubdtype(vals.dtype, "datetime64"):
            return pd.to_datetime(vals).year.values
        # cftime path: extract .year
        try:
            return np.array([v.year for v in vals], dtype=int)
        except Exception:
            # last resort: decode via pandas (may fail for non-Gregorian)
            return pd.to_datetime(vals).year.values

    @staticmethod
    def _to_numeric_time(tcoord: xr.DataArray) -> np.ndarray:
        vals = np.asarray(tcoord.values)
        if np.issubdtype(vals.dtype, np.number):
            return vals.astype(float)
        tt = pd.to_datetime(vals)
        years = tt.year.astype(float)
        doy   = tt.dayofyear.astype(float)
        endy  = pd.to_datetime((tt.year.astype(str) + "-12-31").values)
        denom = np.where(pd.DatetimeIndex(endy).is_leap_year, 366.0, 365.0)
        return years + (doy - 1.0) / denom

    def _ols_slope_over_period(self, da: xr.DataArray, tname: str) -> xr.DataArray:
        """
        Returns slope (units/yr). If use_dask=True, enable parallelized ufunc.
        Note: For performance & correctness, ensure time is a single chunk.
        """
        t = self._to_numeric_time(da[tname])
        tt = xr.DataArray(t, dims=(tname,), coords={tname: da[tname]})

        def _slope(y, x):
            m = np.isfinite(y) & np.isfinite(x)
            if m.sum() < 2:
                return np.nan
            a, _ = np.polyfit(x[m], y[m], 1)
            return a

        if self.use_dask and getattr(da, "chunks", None) is not None:
            da = da.chunk({tname: -1})

        dask_mode = "parallelized" if self.use_dask else "forbidden"
        return xr.apply_ufunc(
            _slope, da, tt,
            input_core_dims=[[tname], [tname]],
            output_core_dims=[[]],
            vectorize=True, dask=dask_mode,
            output_dtypes=[float],
            dask_gufunc_kwargs={"allow_rechunk": True},
        )

    # ---------- aggregation & anomaly ----------
    def _ensure_datetime(self, da: xr.DataArray, tname: str) -> xr.DataArray:
        if np.issubdtype(da[tname].dtype, np.datetime64):
            return da
        try:
            ds_tmp = xr.Dataset({da.name if da.name else "_tmp": da})
            da_dec = xr.decode_cf(ds_tmp)[list(ds_tmp.data_vars)[0]]
            if np.issubdtype(da_dec[tname].dtype, np.datetime64):
                return da_dec
        except Exception:
            pass
        return da

    def _monthly_to_seasonal(self, da: xr.DataArray, tname: str, season: str) -> xr.DataArray:
        """
        Convert monthly series to seasonal/annual means.
        - ANN: calendar-year mean (requires 12 months)
        - DJF: Dec(prev)-Jan-Feb labeled by Jan year (requires all 3 months)
        - MAM/JJA/SON: standard meteorological seasons (3-month means)
        - MON: no aggregation (leave monthly)
        """
        s = season.upper()
        if s == "MON":
            return da

        da = self._ensure_datetime(da, tname)

        if not np.issubdtype(da[tname].dtype, np.datetime64):
            if s != "ANN":
                raise ValueError(
                    f"Season '{s}' requires monthly datetime coordinate; got dtype={da[tname].dtype}."
                )
            g = da.groupby(f"{tname}")
            cnt = g.count(dim=tname)
            out = g.mean(dim=tname, skipna=True).where(cnt >= 1)
            return out

        if s == "ANN":
            g   = da.groupby(f"{tname}.year")
            cnt = g.count(dim=tname)
            out = g.mean(dim=tname, skipna=True).where(cnt >= 12)
            if tname in out.coords:
                out = out.drop_vars(tname)
            non_spatial = [d for d in out.dims if d not in (self.lat_name, self.lon_name)]
            if not non_spatial:
                raise ValueError("No non-spatial dimension found after annual aggregation.")
            year_dim = non_spatial[0]
            years = (out[year_dim].values.astype("int64")
                     if year_dim in out.coords else
                     np.asarray(out.get_index(year_dim), dtype="int64"))
            out = out.rename({year_dim: tname}).assign_coords({tname: years})
            return out

        # Seasonal (DJF/MAM/JJA/SON)
        tvals = xr.DataArray(da[tname].values, dims=(tname,))
        season_year = []
        mask = []
        for ts in pd.to_datetime(tvals.values):
            m = ts.month; y = ts.year
            if   s == "DJF" and (m in (12, 1, 2)):
                sy = y + 1 if m == 12 else y
                sc = "DJF"
            elif s == "MAM" and m in (3, 4, 5):
                sy, sc = y, "MAM"
            elif s == "JJA" and m in (6, 7, 8):
                sy, sc = y, "JJA"
            elif s == "SON" and m in (9, 10, 11):
                sy, sc = y, "SON"
            else:
                sc, sy = None, None
            mask.append(sc is not None)
            season_year.append(sy if sc is not None else -1)
        mask = xr.DataArray(mask, dims=(tname,))
        lab_year = xr.DataArray(season_year, dims=(tname,))

        da2 = da.where(mask, drop=True)
        ly2 = lab_year.where(mask, drop=True)

        g = da2.groupby(ly2)
        cnt = g.count(dim=tname)
        need = 3
        out = g.mean(dim=tname, skipna=True).where(cnt >= need)

        first_dim = list(out.dims)[0]
        out = out.rename({first_dim: tname})
        out = out.assign_coords({tname: np.array(out[tname].values, dtype="int64")})
        return out

    def _fixed_climo_anomaly(self, da: xr.DataArray, tname: str, season: str) -> xr.DataArray:
        """
        Fixed-baseline anomalies.
        - MON: subtract baseline *monthly* climatology (per calendar month).
        - ANN/seasonal: subtract baseline mean of the already-aggregated series.
        """
        if season.upper() == "MON":
            da = self._ensure_datetime(da, tname)

        yrs = self._year_array(da[tname])
        base_mask = (yrs >= self.baseline[0]) & (yrs <= self.baseline[1])
        base_sel = da.where(xr.DataArray(base_mask, dims=(tname,)), drop=True)

        if season.upper() == "MON" and np.issubdtype(da[tname].dtype, np.datetime64):
            clim = base_sel.groupby(f"{tname}.month").mean(tname, skipna=True)
            out = da.groupby(f"{tname}.month") - clim
            out.attrs.update(da.attrs)
            out.attrs["anomaly_baseline"] = f"{self.baseline[0]}–{self.baseline[1]} (monthly climatology)"
            return out

        mu = base_sel.mean(tname, skipna=True)
        out = da - mu
        out.attrs.update(da.attrs)
        out.attrs["anomaly_baseline"] = f"{self.baseline[0]}–{self.baseline[1]}"
        return out

    def _mask_period(self, da: xr.DataArray, tname: str) -> xr.DataArray:
        """
        Mask to the effective analysis window (target_period if provided, else PERIOD),
        intersected with the data's available years.
        """
        yrs = self._year_array(da[tname])
        y_av0, y_av1 = int(np.nanmin(yrs)), int(np.nanmax(yrs))
        y0, y1 = self.analysis_period
        e0, e1 = max(y0, y_av0), min(y1, y_av1)
        if e0 > e1:
            raise ValueError(
                f"No overlap between data [{y_av0},{y_av1}] and analysis window [{y0},{y1}]."
            )
        m = (yrs >= e0) & (yrs <= e1)
        out = da.where(xr.DataArray(m, dims=(tname,)), np.nan)
        out.attrs = dict(out.attrs)
        out.attrs["analysis_period_effective"] = f"{e0}–{e1}"
        return out

    # ---------- region helpers ----------
    def _normalize_lon_bounds(self, lon_min: float, lon_max: float, to360: bool) -> Tuple[float, float]:
        if to360:
            lm = (lon_min + 360.0) % 360.0
            lM = (lon_max + 360.0) % 360.0
        else:
            lm = ((lon_min + 180.0) % 360.0) - 180.0
            lM = ((lon_max + 180.0) % 360.0) - 180.0
        return lm, lM

    def _region_mask(self, ds_for_grid: xr.Dataset, region: Dict[str, Tuple[float, float]]) -> xr.DataArray:
        lat = ds_for_grid[self.lat_name]
        lon = ds_for_grid[self.lon_name]

        lat_min, lat_max = region["lat_range"]
        lon_min, lon_max = region["lon_range"]

        # After coercion, data lon should be -180..180 if flag is True
        data_is_0360 = float(lon.max()) > 180.0  # still handle if coercion is disabled
        lon_vals = lon % 360.0 if data_is_0360 else ((lon + 180.0) % 360.0) - 180.0

        lm, lM = self._normalize_lon_bounds(lon_min, lon_max, to360=data_is_0360)

        def _span(a, b):
            d = (b - a) % 360.0
            return 360.0 if d == 0.0 else d

        span = _span(lm, lM)
        if span >= 360.0 - 1e-9:
            mask_lon = xr.ones_like(lon_vals, dtype=bool)
        else:
            if lm <= lM:
                mask_lon = (lon_vals >= lm) & (lon_vals <= lM)
            else:
                mask_lon = (lon_vals >= lm) | (lon_vals <= lM)

        mask_lat = (lat >= lat_min) & (lat <= lat_max)

        if lon.ndim == 1 and lat.ndim == 1:
            mlat2d, _  = xr.broadcast(mask_lat, lon_vals)
            _, mlon2d  = xr.broadcast(lat,     mask_lon)
            mask = mlat2d & mlon2d
        else:
            mask = mask_lat & mask_lon

        return mask

    def _area_weighted_mean_over_region(self, da: xr.DataArray, ds_for_grid: xr.Dataset,
                                        region: Dict[str, Tuple[float, float]]) -> xr.DataArray:
        lat = ds_for_grid[self.lat_name]
        w = np.cos(np.deg2rad(lat))
        w = xr.broadcast(w, da)[0]

        mask = self._region_mask(ds_for_grid, region)
        mask = xr.broadcast(mask, da)[0]

        w_eff = xr.where(np.isfinite(da) & mask, w, 0.0)
        sum_dims = [d for d in da.dims if d not in ("ens",) and d != self.time_coord]
        den = w_eff.sum(sum_dims)
        den = xr.where(den == 0, np.nan, den)
        return (da * w_eff).sum(sum_dims) / den

    # ---------- I/O ----------
    def _open_ds(self, path: str) -> xr.Dataset:
        ds = xr.open_dataset(path, engine="netcdf4",
                             decode_times=True,
                             chunks="auto" if self.use_dask else None)

        if self.normalize_lon_to_180:
            ds = self._coerce_lon_to_m180_p180(ds)

        # --- quick peek BEFORE normalization ---
        try:
            raw_t = ds[self._detect_time_coord(ds)]
            print("[RAW] time last 3:", raw_t.values[-3:])
            if "time_bnds" in ds:
                tb = ds["time_bnds"]
            elif "time_bounds" in ds:
                tb = ds["time_bounds"]
            elif "bounds_time" in ds:
                tb = ds["bounds_time"]
            else:
                tb = None
            if tb is not None:
                # assume (time,2) or (2,time)
                tb2 = tb.transpose(..., self.time_coord)
                left  = tb2[..., 0].values[-3:]
                right = tb2[..., 1].values[-3:]
                print("[RAW] bnds left last3 :", left)
                print("[RAW] bnds right last3:", right)
        except Exception as e:
            print("[RAW] peek failed:", e)

        # unify monthly time
        try:
            tname = self._detect_time_coord(ds)
            ds = self._normalize_time_monthly(ds, tname, policy=self.normalize_time_policy)
            ds = self._clip_time_to_period(ds, tname)  # <-- enforce PERIOD window
            if self.verbose:
                t0 = pd.to_datetime(ds[tname].values[0])
                t1 = pd.to_datetime(ds[tname].values[-1])
                self._log(f"[INFO] {os.path.basename(path)} time: {t0:%Y-%m-%d} → {t1:%Y-%m-%d} [{self.normalize_time_policy}]")
        except Exception as e:
            self._log(f"[WARN] time normalization skipped for {path}: {e}")

        return ds

    # ---------- single-season per-member ----------
    def process_member(self, path: str, season: str) -> Dict[str, xr.DataArray]:
        """Run the full pipeline for ONE season on one member file."""
        with self._open_ds(path) as ds:
            return self.process_member_for_season(ds, season)

    # ---------- multi-season helpers (reuse single open per member) ----------
    def process_member_for_season(self, ds: xr.Dataset, season: str) -> Dict[str, xr.DataArray]:
        """
        Run the full pipeline for ONE season using an already-open Dataset 'ds'.
        This avoids re-opening the same file for each season.
        """
        s = season.upper()
        tname = self._detect_time_coord(ds)
        da = ds[self.var_name]

        # monthly -> seasonal aggregation
        da_seas = self._monthly_to_seasonal(da, tname, season=s)

        if s == "ANN" and self.verbose:
            yrs = self._year_array(da_seas[tname])
            self._log(f"[DEBUG] ANN years span: {int(np.nanmin(yrs))} … {int(np.nanmax(yrs))}")

        # anomalies & analysis window
        da_an  = self._fixed_climo_anomaly(da=da_seas, tname=tname, season=s)
        da_win = self._mask_period(da_an, tname)

        # Optional: spatial-only rechunk for performance (keep time consolidated)
        if self.use_dask:
            da_win = da_win.chunk({self.lat_name: 64, self.lon_name: 64, tname: -1})

        # β_global over region_global (denominator)
        gmean   = self._area_weighted_mean_over_region(da_win, ds, self.region_global)
        beta_g  = self._ols_slope_over_period(gmean, tname)

        # β_local field and trend/AA
        beta_l  = self._ols_slope_over_period(da_win, tname)   # K/yr per grid cell
        trend   = beta_l * 10.0                                 # K/decade

        # Stable epsilon handling for tiny |β_global|
        beta_g_val = float(beta_g.values)
        denom = beta_g + np.copysign(self.aa_eps, beta_g_val)
        aa = xr.where(np.abs(beta_g_val) < self.aa_eps, np.nan, beta_l / denom)

        # Mask outputs to region_local
        mask_loc = self._region_mask(ds, self.region_local)
        mask_loc = xr.broadcast(mask_loc, beta_l)[0]
        beta_l = beta_l.where(mask_loc)
        trend  = trend.where(mask_loc)
        aa     = aa.where(mask_loc)

        # ensure coords/attrs consistency
        for arr in (beta_l, trend, aa):
            if self.lat_name in ds.coords:
                arr.coords[self.lat_name].attrs.update(ds[self.lat_name].attrs)
            if self.lon_name in ds.coords:
                arr.coords[self.lon_name].attrs.update(ds[self.lon_name].attrs)

        eff_str = da_win.attrs.get(
            "analysis_period_effective",
            f"{self.analysis_period[0]}–{self.analysis_period[1]}",
        )

        beta_l.attrs["units"] = "K/yr"
        trend .attrs["units"] = "K/decade"
        aa    .attrs.update({
            "units": "1",
            "definition": (
                f"β_local/β_global over {eff_str} "
                f"(baseline {self.baseline[0]}–{self.baseline[1]}, season={s}; "
                f"β_global region={self.region_global}; β_local region={self.region_local})"
            ),
        })

        out = {"beta_local": beta_l, "trend_dec": trend, "aa_map": aa, "beta_global": beta_g}

        if self.emit_aa_index:
            try:
                # area-weighted mean of β_local within region_local, then divide by β_global
                grid_ds = xr.Dataset({self.lat_name: ds[self.lat_name], self.lon_name: ds[self.lon_name]})
                num = self._area_weighted_mean_over_region(beta_l, grid_ds, self.region_local)
                aa_index = num / beta_g
            except Exception:
                aa_index = xr.DataArray(np.nan)
            out["aa_index"] = aa_index

        return out


    def streamed_mean(
        self,
        paths: List[str],
        season: str,
        keep_members: bool = False,
    ):
        """Single-season ensemble mean/optionally members (explicit `season`)."""
        sum_beta_l = sum_trend = sum_aa = None
        sum_beta_g = 0.0
        sum_aa_index = 0.0
        have_aa_index = False

        count = 0
        members = []

        for i, p in enumerate(paths):
            self._log(f"[{i+1}/{len(paths)}] {os.path.basename(p)}")
            md = self.process_member(p, season=season)
            count += 1
            if sum_beta_l is None:
                sum_beta_l = md["beta_local"].astype("f8")
                sum_trend  = md["trend_dec"].astype("f8")
                sum_aa     = md["aa_map"].astype("f8")
            else:
                sum_beta_l = sum_beta_l + md["beta_local"]
                sum_trend  = sum_trend  + md["trend_dec"]
                sum_aa     = sum_aa     + md["aa_map"]

            sum_beta_g += float(md["beta_global"].values)

            if "aa_index" in md:
                have_aa_index = True
                sum_aa_index += float(md["aa_index"].values)

            if keep_members:
                mem = {k: v.expand_dims({"ens": [count]}) for k, v in md.items()}
                members.append(mem)

        mean_beta_l = sum_beta_l / count
        mean_trend  = sum_trend  / count
        mean_aa     = sum_aa     / count
        mean_beta_g = sum_beta_g / count
        if have_aa_index:
            mean_aa_index = sum_aa_index / count

        ds_mean = xr.Dataset(
            data_vars={
                "beta_local": mean_beta_l,
                "trend_dec":  mean_trend,
                "aa_map":     mean_aa,
                "beta_global": xr.DataArray(mean_beta_g, attrs={"units": "K/yr"}),
                **({"aa_index": xr.DataArray(mean_aa_index)} if have_aa_index else {}),
            },
            attrs={
                "ensemble_mean": True,
                "file_period_start": self.period[0], "file_period_end": self.period[1],
                "analysis_start": self.analysis_period[0], "analysis_end": self.analysis_period[1],
                "baseline_start": self.baseline[0], "baseline_end": self.baseline[1],
                "group": self.group, "var_name": self.var_name, "season": season,
                "region_global": str(self.region_global),
                "region_local":  str(self.region_local),
            }
        )

        ds_members = None
        if keep_members and members:
            ds_members = xr.Dataset()
            for key in ("beta_local", "trend_dec", "aa_map"):
                ds_members[key] = xr.concat([m[key] for m in members], dim="ens")
            ds_members["beta_global"] = xr.concat([m["beta_global"] for m in members], dim="ens")
            if have_aa_index:
                ds_members["aa_index"] = xr.concat([m["aa_index"] for m in members], dim="ens")
            ds_members.attrs.update(ds_mean.attrs)
            ds_members.attrs["ensemble_mean"] = False

        return ds_mean, ds_members

    def streamed_mean_multi(
        self,
        paths: List[str],
        seasons: List[str],
        keep_members: bool = False,
    ):
        """
        Open each member ONCE, compute diagnostics for ALL seasons, and
        return a dict: season -> (ds_mean, ds_members).
        """
        seasons = [s.upper() for s in seasons]
        acc = {
            s: {
                "sum_beta_l": None, "sum_trend": None, "sum_aa": None,
                "sum_beta_g": 0.0, "sum_aa_index": 0.0, "have_aa_index": False,
                "count": 0, "members": []
            } for s in seasons
        }

        for i, p in enumerate(paths):
            self._log(f"[{i+1}/{len(paths)}] {os.path.basename(p)} (multi-season)")
            with self._open_ds(p) as ds:
                for s in seasons:
                    md = self.process_member_for_season(ds, s)
                    A = acc[s]
                    A["count"] += 1

                    if A["sum_beta_l"] is None:
                        A["sum_beta_l"] = md["beta_local"].astype("f8")
                        A["sum_trend"]  = md["trend_dec"].astype("f8")
                        A["sum_aa"]     = md["aa_map"].astype("f8")
                    else:
                        A["sum_beta_l"] = A["sum_beta_l"] + md["beta_local"]
                        A["sum_trend"]  = A["sum_trend"]  + md["trend_dec"]
                        A["sum_aa"]     = A["sum_aa"]     + md["aa_map"]

                    A["sum_beta_g"] += float(md["beta_global"].values)

                    if "aa_index" in md:
                        A["have_aa_index"] = True
                        A["sum_aa_index"] += float(md["aa_index"].values)

                    if keep_members:
                        A["members"].append({k: v.expand_dims({"ens": [A["count"]]}) for k, v in md.items()})

        out: Dict[str, Tuple[xr.Dataset, Optional[xr.Dataset]]] = {}
        for s in seasons:
            A = acc[s]
            c = max(A["count"], 1)
            mean_beta_l = A["sum_beta_l"] / c
            mean_trend  = A["sum_trend"]  / c
            mean_aa     = A["sum_aa"]     / c
            mean_beta_g = A["sum_beta_g"] / c
            if A["have_aa_index"]:
                mean_aa_index = A["sum_aa_index"] / c

            ds_mean = xr.Dataset(
                data_vars={
                    "beta_local": mean_beta_l,
                    "trend_dec":  mean_trend,
                    "aa_map":     mean_aa,
                    "beta_global": xr.DataArray(mean_beta_g, attrs={"units": "K/yr"}),
                    **({"aa_index": xr.DataArray(mean_aa_index)} if A["have_aa_index"] else {}),
                },
                attrs={
                    "ensemble_mean": True,
                    "file_period_start": self.period[0], "file_period_end": self.period[1],
                    "analysis_start": self.analysis_period[0], "analysis_end": self.analysis_period[1],
                    "baseline_start": self.baseline[0], "baseline_end": self.baseline[1],
                    "group": self.group, "var_name": self.var_name, "season": s,
                    "region_global": str(self.region_global),
                    "region_local":  str(self.region_local),
                }
            )

            ds_members = None
            if keep_members and A["members"]:
                ds_members = xr.Dataset()
                for key in ("beta_local", "trend_dec", "aa_map"):
                    ds_members[key] = xr.concat([m[key] for m in A["members"]], dim="ens")
                ds_members["beta_global"] = xr.concat([m["beta_global"] for m in A["members"]], dim="ens")
                if A["have_aa_index"]:
                    ds_members["aa_index"] = xr.concat([m["aa_index"] for m in A["members"]], dim="ens")
                ds_members.attrs.update(ds_mean.attrs)
                ds_members.attrs["ensemble_mean"] = False

            out[s] = (ds_mean, ds_members)

        return out

    # ---------- save & plot ----------
    @staticmethod
    def save_nc(ds_out: xr.Dataset, out_path: str, *, compress=True, complevel=4):
        import numpy as np
        os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

        def _coerce_attr(v):
            # Booleans -> uint8; numpy scalars -> Python scalars; complex containers -> str
            if isinstance(v, (np.bool_, bool)):
                return np.uint8(1) if v else np.uint8(0)
            if isinstance(v, (np.integer, np.floating)):
                return v.item()
            if isinstance(v, (list, tuple, dict, set)):
                return str(v)
            return v

        # Shallow copy and sanitize attrs (dataset + each var)
        ds = ds_out.copy(deep=False)
        ds.attrs = {k: _coerce_attr(v) for k, v in dict(ds_out.attrs).items()}
        for k in ds.data_vars:
            ds[k].attrs = {ak: _coerce_attr(av) for ak, av in dict(ds_out[k].attrs).items()}

        enc = {v: {"zlib": bool(compress), "complevel": int(complevel), "shuffle": True, "dtype": "f4"}
               for v in ds.data_vars}
        ds.to_netcdf(out_path, encoding=enc)
        return out_path


In [15]:
if __name__ == "__main__":
    DATA_DIR  = f"{DATA_DIR}"
    OUT_DIR   = DIAG_DIR
    os.makedirs(OUT_DIR, exist_ok=True)
    
    # ---- Pair experiments consistently (order matters) ----
    EXPS    = ["ERA5",        "NOAA_20C",    "E3SMv3-LE"]
    GROUP   = ["ERA5",        "NOAA_20C",    "v3.LR.historical"]
    PERIOD  = [(1979, 2018),  (1853, 2015),  (1850, 2050)]   # ERA5 first, E3SM second
    NENS    = [1,             1,             25]

    PTARGT  = (1979, 2014)    
    PCLIMO  = (1981, 2010)
    VAR     = "TS"

    region_dict = {
        "arctic": {"name": "Arctic2", "lat_range": (66.5, 90.0),  "lon_range": (-180, 180)},
        "global": {"name": "global", "lat_range": (-90.0, 90.0), "lon_range": (-180, 180)},
    }

    lat_name   = "lat"
    lon_name   = "lon"
    time_coord = None
    aa_eps     = 1e-5
    region_subdir = "global"
    verbose    = True
    use_dask   = True
    chunks     = {"lat": 64, "lon": 64}  # optional; class opens with chunks="auto"
    write_members = True
    
    # NOTE: If your ERA5 file does NOT have ".en00." in its name,
    # switch to: file_template = "{group}.monthly_base.{y0m}-{y1m}.nc"
    file_template = "{group}.%(ENS).monthly_base.{y0m}-{y1m}.nc"

    keep_members = True
    SEASONS = ["ANN", "DJF", "MAM", "JJA", "SON"]
    SEASONS = [s.upper() for s in SEASONS]

    if use_dask:
        xr.set_options(keep_attrs=True)
        
    for exp, grp, prd, nens in zip(EXPS, GROUP, PERIOD, NENS):
        print(f"\n[RUN] {exp} | group={grp} | period={prd[0]}-{prd[1]} | nens={nens}")

        diag = ArcticAmplificationDiagnostics(
            data_dir=DATA_DIR,
            group=grp,
            period=prd,
            baseline=PCLIMO,
            target_period=PTARGT,
            var_name=VAR,
            nens=nens,
            lat_name=lat_name,
            lon_name=lon_name,
            time_coord=time_coord,
            aa_eps=aa_eps,
            region_global=region_dict["global"],
            region_local=region_dict["arctic"],
            region_subdir=region_subdir,
            verbose=verbose,
            use_dask=use_dask,
            chunks=chunks,
            file_template=file_template,
        )

        paths = diag.discover_paths()
        if len(paths) < nens:
            print(f"[WARN] Found {len(paths)}/{NENS} members; continuing with available files.")

        # Open each member ONCE and compute all seasons
        results = diag.streamed_mean_multi(
            paths,
            seasons=SEASONS,
            keep_members=keep_members
        )

        # Save ensemble means (and members container if requested)
        for seas, (ds_mean, ds_members) in results.items():
            tag = f"{exp}_{grp}_{VAR}_{seas}_arctic_{PTARGT[0]}-{PTARGT[1]}"
            diag.save_nc(ds_mean, os.path.join(OUT_DIR, f"AA2_{tag}.mean.nc"))
            if ds_members is not None:
                diag.save_nc(ds_members, os.path.join(OUT_DIR, f"AA2_{tag}.members.nc"))
            print(f"[OK] {seas} done.")



[RUN] ERA5 | group=ERA5 | period=1979-2018 | nens=1
[INFO] Found 1 ensemble members.
[1/1] ERA5.en00.monthly_base.197901-201812.nc (multi-season)
[RAW] time last 3: ['2018-10-01T00:00:00.000000000' '2018-11-01T00:00:00.000000000'
 '2018-12-01T00:00:00.000000000']
[INFO] ERA5.en00.monthly_base.197901-201812.nc time: 1979-01-01 → 2018-12-01 [left]
[DEBUG] ANN years span: 1979 … 2018
[OK] ANN done.
[OK] DJF done.
[OK] MAM done.
[OK] JJA done.
[OK] SON done.

[RUN] NOAA_20C | group=NOAA_20C | period=1853-2015 | nens=1
[INFO] Found 1 ensemble members.
[1/1] NOAA_20C.en00.monthly_base.185301-201512.nc (multi-season)
[RAW] time last 3: ['2015-10-01T00:00:00.000000000' '2015-11-01T00:00:00.000000000'
 '2015-12-01T00:00:00.000000000']
[INFO] NOAA_20C.en00.monthly_base.185301-201512.nc time: 1853-01-01 → 2015-12-01 [left]
[DEBUG] ANN years span: 1853 … 2015
[OK] ANN done.
[OK] DJF done.
[OK] MAM done.
[OK] JJA done.
[OK] SON done.

[RUN] E3SMv3-LE | group=v3.LR.historical | period=1850-2050 | n

[RAW] time last 3: ['2050-10-01T00:00:00.000000000' '2050-11-01T00:00:00.000000000'
 '2050-12-01T00:00:00.000000000']
[INFO] v3.LR.historical.en22.monthly_base.185001-205012.nc time: 1850-01-01 → 2050-12-01 [left]
[DEBUG] ANN years span: 1850 … 2050
[24/25] v3.LR.historical.en23.monthly_base.185001-205012.nc (multi-season)
[RAW] time last 3: ['2050-10-01T00:00:00.000000000' '2050-11-01T00:00:00.000000000'
 '2050-12-01T00:00:00.000000000']
[INFO] v3.LR.historical.en23.monthly_base.185001-205012.nc time: 1850-01-01 → 2050-12-01 [left]
[DEBUG] ANN years span: 1850 … 2050
[25/25] v3.LR.historical.en24.monthly_base.185001-205012.nc (multi-season)
[RAW] time last 3: ['2050-10-01T00:00:00.000000000' '2050-11-01T00:00:00.000000000'
 '2050-12-01T00:00:00.000000000']
[INFO] v3.LR.historical.en24.monthly_base.185001-205012.nc time: 1850-01-01 → 2050-12-01 [left]
[DEBUG] ANN years span: 1850 … 2050
[OK] ANN done.
[OK] DJF done.
[OK] MAM done.
[OK] JJA done.
[OK] SON done.
